<a href="https://colab.research.google.com/github/avinashmane/runpix-nb/blob/master/run_pix_manage_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Manage RUN PIX


## config

In [9]:
import yaml
cfg=yaml.safe_load(r"""

AUTH_MODE: svc_ac   # possible values:-->>> user or service_account
web_url: https://run-pix.web.app/
sheets:
  mychoice:
    url: https://docs.google.com/spreadsheets/d/1Qsm_y3Dz8RQwvG2YDuEGlKHlpJRLXV__CFthV8j_ATA/edit#gid=88995822
    range:
    columnMap:

tz: Asia/Kolkata
d_t_fmt: "%a %b %e %Y %H:%M:%S %z"

files:
  firebase_service_account_file_id: '1J_Le9f_pG8qXTAUPl9TLVNlVcuLNkFEs'
""")

from google.colab import userdata
cfg['svc_ac']=yaml.safe_load(userdata.get('runpix_sa'))



## HOW TO USE

* Menu ->  Runtime > Run All
* Scroll down to Overall Dashboard
  * Copy race from previous (Check inputs first)
  * Load Race
  * Load Bibs ((Check all bibs are uploaded)
  * Export results
## Features
* Copy bib master to race start list
* copy results My Choice sheet
  * https://docs.google.com/spreadsheets/d/1Qsm_y3Dz8RQwvG2YDuEGlKHlpJRLXV__CFthV8j_ATA/edit#gid=1192889383
* delete Raceimages
jump to relevant section after this

## Ref
* https://docs.gspread.org/en/v5.7.1/

# Code


In [25]:
"Setup"
# Import PyDrive and associated libraries.
# This only needs to be done once per notebook.
from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials
from tqdm.notebook import tqdm


# Authenticate and create the PyDrive client.
# This only needs to be done once per notebook.
if cfg['AUTH_MODE']=="svc_ac":
  import google.auth
  cred=google.auth.load_credentials_from_dict(cfg['svc_ac'])

gauth = GoogleAuth()
gauth.credentials = cred
drive = GoogleDrive(gauth)

# following 3 lines for spreadsheet read/write
import gspread
gc = gspread.service_account_from_dict(cfg['svc_ac'])

# # Download a file based on its file ID.
# # A file ID looks like: laggVyWshwcyP6kEI-y_W3P8D26sz
# def getDriveIdContent(file_id = 'REPLACE_WITH_YOUR_FILE_ID'):
#   downloaded = drive.CreateFile({'id': file_id})
#   return downloaded.GetContentString()
# def copyDriveIdtoFile(file_id = 'REPLACE_WITH_YOUR_FILE_ID',dest_path=""):
#   downloaded = drive.CreateFile({'id': file_id})
#   return downloaded.GetContentFile(dest_path)

AttributeError: 'tuple' object has no attribute '__module__'

In [36]:
gc.list_spreadsheet_files()

[{'id': '1eWB8KQkfsx8TVib8m8RNlv1PZNgoBcUCBt02lMhAPYA',
  'name': 'Townscript 2023 complete',
  'createdTime': '2022-12-23T16:36:09.333Z',
  'modifiedTime': '2024-03-14T18:43:37.332Z'}]

In [ ]:
copyDriveIdtoFile(cfg['files']['firebase_service_account_file_id'],'firebase_service_account.json')
# print("Downloaded content \"{getDriveIdContent('1J_Le9f_pG8qXTAUPl9TLVNlVcuLNkFEs')}\"".format())

In [37]:
"""  INSTALL of modules
"""
required_modules="firebase-admin itables pydash".split()

allmodules=!pip list
allmodules=[_.split()[0] for _ in allmodules]
for mod in required_modules:
  if not mod in allmodules:
    !pip install firebase_admin
!pip install pydash

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.9/101.9 kB 2.5 MB/s eta 0:00:00


In [38]:
%%writefile util.py
import os
from google.colab import data_table
data_table.enable_dataframe_formatter()

from datetime import datetime
from IPython.display import display, Markdown,Image
import pydash as py_

def getFileName(file):
    return file.split("\\")[-1]
def getFileIsoDate(file):
    return datetime.fromtimestamp( os.path.getctime(file) ).isoformat()
def concatDict(dict1,dict2):
  dict1.update(dict2)
  return dict1
def dir2df(rootPath):
    arr=[]
    for root, dirs, files in os.walk(rootPath, topdown=False):
        for name in files:
            arr.append({'type':'file', "root":root, 'path':name})
        for name in dirs:
            arr.append({'type':'dir', "root":root, 'path':name})
    return pd.DataFrame(arr)

def getObjAttr(*attrs):
    return {attr: getattr(x, attr) for attr in attrs}

Writing util.py


## main

In [41]:
from firebase_admin import firestore, storage, credentials
credentials??

In [43]:
"  MAIN CODE "
import os,importlib
from datetime import datetime
import pandas as pd
from IPython import display
from IPython.display import display, Markdown,Image
import ipywidgets as widgets
from ipywidgets import Layout, Button, Box, VBox,HBox
from functools import lru_cache
import firebase_admin
from firebase_admin import firestore, storage, credentials
import pydash as py_
" Local includes"
from util import getFileName, getFileIsoDate, dir2df,concatDict
def md(*args): display(Markdown(args))

class Firestore:
    def __init__(self,
                 service_account_file=cfg['svc_ac'],#"firebase_service_account.json",
                 bucket_name='run-pix.appspot.com'):
        cred = credentials.Certificate(service_account_file)
        try: self.app = firebase_admin.initialize_app(cred)
        except: self.app = firebase_admin.get_app()
        self.fstore=firestore.client()
        self.storage=firebase_admin.storage
        self.bucket=firebase_admin.storage.bucket(name=bucket_name, app=None)
    def getDoc(self,path): #even
        return self.fstore.document(path).get().to_dict()
    def getDocs(self,path): #odd
        ret_d=[]
        docs = runpix.fstore.collection(path).stream()
        for i,doc in enumerate(docs):
            ret_d.append(concatDict(  {"id":doc.id}, doc.to_dict()))
        return ret_d

    def updateDoc(self,path,values):
        return self.fstore.document(path).update(values)

    def collection(self):
        return self.fstore

    def fs2Df(self,collPath):
        def _f(x): # dict --> array of dicts
            try:
              _=concatDict({"id":x.id,'ref':x},x.get().to_dict())
            except:
              _={"id":x.id,'ref':x}
            return _
        return pd.DataFrame([_f(_) for _ in self.fstore.collection(collPath).list_documents()])
    "delete all subnodes"
    def delete_all_docs(self,collection):
        for n in collection.get():
            print(n.__dict__)
            n._reference.delete()

runpix=Firestore()

bucket=firebase_admin.storage.bucket(name='run-pix.appspot.com', app=None)

class Race:
    storage_prefixes='processed uploads thumbs'.split()
    status=[]
    blobs={}
    bibs=[]
    images=[]
    readings=[]
    def __init__(self,id):
        print(f"Race id: {id}")
        self.fstore=firestore.client()
        self.bucket=firebase_admin.storage.bucket(name='run-pix.appspot.com', app=None)
        self.id = id
        for k,v in self.fstore.document(f'races/{id}').get().to_dict().items():
          setattr(self,k,v)
    def __repr__(self):
        return f"""<Race:{self.id} bibs:{len(self.bibs)} Images:{len(self.images)} readings:{len(self.readings)}
        status: {self.status}
        blobs:{[(_,len(self.blobs[_].keys())) for _ in self.blobs.keys()]}>
        """

    def copyRace(self,toId):
        """
        COPY RACE

        # copyRace('mychoice23apr','mychoice23mar')
        # copyRace('mychoice23mar','mychoice23jan')
        """
        fromRace=runpix.fstore.document(f"races/{self.id}").get().to_dict()
        print(fromRace)
        newDocRef=runpix.fstore.document(f"races/{toId}")
        # print(newDocRef,newDocRef.get().exists)
        if newDocRef.get().exists:
          raise Exception(f"Race id {toId} already exists ")
        else:
          return newDocRef.set(fromRace)


    def getBlobs(self):
        for folder in self.storage_prefixes:
            self.blobs[folder]={b.id:{"id":b.id,"name":b.name,"blob":b,"size":b.size}
                                    for b in bucket.list_blobs(prefix=f"{folder}/{self.id}")}
        return self.blobs
        # self.blobs_df=pd.DataFrame(
    @lru_cache
    def getBibs(self):
        # df_d=[]
        #     # .where(u'textAnnotations', u'==', [])
        # docs = runpix.fstore.collection(u'races',self.id,'bibs').stream()
        # for i,doc in enumerate(docs):
        #     df_d.append(concatDict(  {"id":doc.id}, doc.to_dict()))
        #     # self.bibs=[_.get().to_dict() for _ in self.fstore.collection(f'races/{self.id}/bibs').list_documents()]
        self.bibs=runpix.getDocs(f'races/{self.id}/bibs')
        return self.bibs
    @lru_cache
    def getReadings(self):
        self.readings=runpix.getDocs(f'races/{self.id}/readings')
        return self.readings
    def getReadingsDf(self):
        df=pd.DataFrame(self.getReadings())
        df['rank']=df.groupby(['waypoint', 'bib'])['timestamp'].rank(ascending=True)
        def get_st(rec,rank='rank'):
          if (rec['status']==rec['status'] ) and (rec['status']!='noname'):
            return rec['status']
          if rec[rank]==1:
            if rec['bib']!=rec['bib']:
              return rec['id'].split('_')[1]
            return 'valid'
          elif rec[rank]==rec[rank]:
            return 'xdup'
          else:
            return 'misc'

        df['st']=df.apply(get_st,axis=1)
        return df

    @lru_cache
    def getResult(self):
        return runpix.getDocs(f'races/{self.id}/result')

    def getimages(self,n=None):
        df_d=[]
            # .where(u'textAnnotations', u'==', [])
        docs = runpix.fstore.collection(u'races',self.id,'images').stream()
        for i,doc in enumerate(docs):
            data_d=doc.to_dict()
            ob=data_d
            df_d.append(ob)
            if n!=None and i>n:
                break
        self.images=df_d
        return df_d
    def saveBib(self,bibData):
      _bibData=bibData if isinstance (bibData,dict) else bibData.to_dict()
      _bibData['Bib']=str(_bibData['Bib'])
      _bibData['Name']=_bibData['Name'].upper()
      _path=f"races/{self.id}/bibs/{_bibData['Bib']}"
      runpix.fstore.document(_path).set(_bibData)
      print(f"loaded {_bibData['Bib']} {_bibData['Name']}")

    def deleteBlobs(self,folders=['thumbs']):
        for folder in folders:
            for b in race.blobs[folder]:
                try:
                    runpix.bucket.delete_blob(race.blobs[folder][b]['name'])
                    print(race.blobs[folder][b]['name'])
                except:
                    print(f"Error deleting {race.blobs[folder][b]['name']}")

    def get_duration(self,timestamp):
      " Duration since start "
      start_timestamp=race.timestamp['start']
      startDelta=Race.calc_time('2000-01-01T00:00:00')-Race.calc_time(start_timestamp)
      return (Race.calc_time(timestamp)+startDelta).strftime('%X')

    @staticmethod
    def calc_time(reading_timestamp,tz='Asia/Kolkata'):
      try:
          if reading_timestamp[-1]=='Z':
            return pd.Timestamp(reading_timestamp,tz=tz)
          else:
            return pd.Timestamp(reading_timestamp,tz=tz)
      except Exception as e:
        print (f"error {e!r} {reading_timestamp}")
        return pd.NaT


In [49]:
runpix.app

## Dashboard

In [44]:
def dash():
  races_d=runpix.getDocs('races')

  races=py_.chain(races_d).sort(key=lambda x: x['Date'],reverse=True
            ).map(lambda x:(x['id']+' : '+x['Date'],x['id']),
                            ).value()
  display(Markdown("# Race Management"))
  print(f"Total races: {len(races_d)}")
  race_ui=widgets.Dropdown(options=races,value=races[2][1])

  def display_race(race_id):
    global race
    def raceProp(fld): return py_.get(race,fld,'-')

    race= Race(race_id)

    print(f"\nLoc: {raceProp('Location')}\n{raceProp('stats')}\n{raceProp('timestamp')}\n"),
    display(Markdown(f"[RunPix {race_id}]({cfg['web_url']}/r/{race_id})"),

            )

    def display_readings():
      data=race.getReadingsDf()
      display(f"Entries: {data.shape[0]}, Bibs: {data.bib.count()}, Unique Bibs: {data.bib.nunique(dropna=True)}")
      data['dur']=data.timestamp.apply(race.get_duration)
      display(data)

    def display_result():
      data=pd.DataFrame(race.getResult())
      data.drop(['Start Time'], axis=1)
      print(data.shape)
      display(data)

    def display_bibs():
      data=pd.DataFrame(race.getBibs())
      df_bibs=pd.DataFrame(race.getResult())
      print(df_bibs.shape[0],'/',data.shape[0])
      df=data.merge(df_bibs[
                          ['Bib','Category','Race Time']],
                         how='left',
                         on="Bib")
      display(df)

    display( widgets.VBox([widgets.interactive(display_bibs,
                             {'manual': True,'manual_name': f'BIBS {race_id}'},
                             ),
                           widgets.interactive(display_readings,
                             {'manual': True,'manual_name': f'READINGS {race_id}'},
                             ),
                           widgets.interactive(display_result,
                             {'manual': True,'manual_name': f'RESULTS {race_id}'},
                             )]))

  return widgets.interactive(display_race,
                             {'manual': True,'manual_name':'LOAD RACE'},
                             race_id=race_ui,)


### Test placeholder

In [46]:
# del(race)


# Overall Dashboard

In [ ]:
"Dashboard for checking out races"
dash()

In [ ]:
raise Exception('End of normal run here')




# New Section

In [ ]:
"offset to different distance start"
pd.Timestamp(race.timestamp['start_3K'])-pd.Timestamp(race.timestamp['start'])

# END regular

In [ ]:
align waypoint to bib

In [ ]:
readings.pivot_table(index=readings.bib.str[:1],columns='waypoint',
                     aggfunc="count",values='bib').fillna('')

In [ ]:
#readings.query("bib.isnull() | bib.astype(str)[:1]=='3'")
def overrideWayptInReading(reading_rec,wpt):
  rd= runpix.fstore.document(f"races/{race.id}/readings/{reading_rec.id}").get().to_dict()
  if rd['waypoint']!=wpt:
    return runpix.fstore.document(f"races/{race.id}/readings/{reading_rec.id}").update({'waypoint':wpt})
def overrideAllWayptInReadings(bibPattern,wpt)  :
  return readings.loc[readings.bib.fillna('').str.contains(bibPattern),:].apply(lambda rec: overrideWayptInReading(rec,wpt),axis=1)


overrideAllWayptInReadings('^3','3K')
overrideAllWayptInReadings('^5','5K')

# Purge result 10K werun24

In [ ]:
"delete 5k 3k results"
# runpix.fstore.document("races/werun2024").get().to_dict()

"result in dataframe"
result=pd.DataFrame([doc.to_dict() for doc in runpix.fstore.collection(f"races/{race.id}/result").stream()])
display(race)
result

In [ ]:
"delete result for following"
cond="Race.isin(['3K'])"
def deleteBibFromResult(bib):
  return runpix.fstore.document(f"races/{race.id}/result/{bib}").delete()

result.query(cond).Bib.apply(deleteBibFromResult )

## Analysis final result

In [ ]:
raceId = raceId_w.value
oldRaceTab = 'Race_MyCHOICE 2023-10' #@param {type:"string"}

In [ ]:
"""
for every image uploaded manually (type=na)
  for every bib
    that is valid as per list (may not not required)
    if creationtime exists in exif
      validation of time (5k,10k) only finishline photographs
        (25-40min 5k)(45min-130min 10k) men
        (25-55min 5k)(55min-130min 10k) women
        cutoff needed for other photos
      if incorrect timing was updated
        delete incorrect timing
        create new timing

"""
def checkImageReading(txt,img):
    try:
      # (txt,img['timestamp'],img['imagePath'][-4:],img['imagePath'][:30],img['metadata']['DateCreated'])
      if img['metadata']['DateCreated']:
        path=f"races/{race.id}/readings/{img['timestamp']}_{txt}"
        if runpix.fstore.document(path).get().exists:
          print(path,'exists')
        else:
          # print(path,'not exists')
          pass
    except Exception as e:
      print(txt,img['timestamp'],img['imagePath'][-4:],img['imagePath'],img['metadata'])
      print(f"error {e!r}")

for img in race.images:
  for txt in img['texts']:
    checkImageReading(txt,img)

In [ ]:
# race.images

# Race Analysis

In [ ]:
" Analysis of  images.texts  - get data"
def analysisData(race):
    df_bibs=pd.DataFrame(race.bibs)
    df_bibs.shape

    selected_keys="timestamp imagePath texts".split()
    def getImageDtl(img):
      subset= {_:img[_] for _ in selected_keys}
      subset.update({_:img['metadata'][_] for _ in img['metadata'].keys() if 'Date' in _})
      return subset

    df_images=pd.DataFrame(map(getImageDtl,race.images))#[selected_keys]

    return {'bibs':df_bibs,
            'images':df_images}

ana=analysisData(race)
race.id
# ana['images'][:2]

In [ ]:
" Analysis of  images.texts  - report 1"
def anaImages(df_images):
  images_with_scans=df_images.texts.apply(lambda x: len(x) if isinstance(x,list) else 0).value_counts().sort_index()
  display(Markdown("## Analysis of Images"))
  print(f"""Total images: {len(df_images)}
  Images with texts: {len(df_images)-images_with_scans[0]}
  Images with bibs distribution without bib check:
  {images_with_scans}
  """)

anaImages(ana['images'])


In [ ]:
""" Exploded list with bib """
ana['images_bib']=ana['images'].set_index("imagePath").explode(['texts']
                      ).merge(ana['bibs'],left_on='texts',right_on='Bib',how='left')
ana['images_bib'].shape


In [ ]:
display(Markdown("""##  Runners with most photos  """))
ana['images_bib'].sort_values('Name').drop_duplicates(["timestamp"],keep='last'
     ).groupby('Name Bib'.split())['timestamp'].count().sort_values(ascending=False
    )

In [ ]:

ana['bib_img']=ana['bibs'].merge(ana['images'].explode(['texts']),right_on='texts',left_on='Bib',how='inner')
# df_images_bib.sort_values('Name').drop_duplicates(["timestamp"],keep='last'
#      ).groupby('Name Bib'.split())['timestamp'].count().sort_values(ascending=False
#     )
display(Markdown(f"""##  Start list found in the photos
{len(ana['bib_img'].drop_duplicates("Name").texts.values) } / {len(ana['bibs'])}
{len(ana['bib_img'].drop_duplicates("Name").texts.values) *100 / len(ana['bibs']):0.2f}%
"""))


## AdHoc

## check readings for images

In [ ]:
" get all timestamps from the image record"
print(race.id,"images",len(race.images),'readings',len(race.readings),)
def getObjSubset(x,attrs):
    try:
      return {attr: x[attr] for attr in attrs}
    except:
      print("error",x.keys())
# display(race.images[:1])
def getTimestampFromImg(img):
  timestamps=getObjSubset(img,['imagePath','timestamp'])
  timestamps.update({exif:img['metadata'][exif] for exif in img['metadata'].keys() if 'Date' in exif})
  return timestamps
pd.DataFrame([getTimestampFromImg(_) for _ in race.images[:2]])

# Timing processing

In [ ]:
# race.readings
from tqdm import tqdm
def calc_time(x,tz='Asia/Kolkata'):
  try:
      if x[-1]=='Z':
        return pd.Timestamp(x,tz=tz)
      else:
        return pd.Timestamp(x,tz=tz)
  except Exception as e:
    print (f"error {e!r} {x}")
    return pd.NaT
race.start=calc_time(race.timestamp['start'])
Markdown(f"## Race id {race.id} Start : {race.start}")

class mychoice_result:
  cols_in_mychoice="Status	SendTime	email	bib	Split	Time"
  cols="type imagePath userId		bib	waypoint 	timestamp".split()

  row_number=15
  startcol=1
  def __init__(self,race,resultTab):
    self.resultTab=resultTab
    self.race=race
    display(Markdown(f"### Updating {race.id} into tab {resultTab}"))
    self.readTiming()

  def _localeTimestamp(self,dt):
      return str(dt.tz_convert(cfg['tz']).strftime(cfg['d_t_fmt']))
  def readTiming(self):
    " df_tim = READINGS+bibs "
    self.race.getBibs()

    df_tim=pd.DataFrame(self.race.readings
                        ).merge(pd.DataFrame(
                            self.race.bibs),
                                left_on='bib',right_on='Bib',
                                how='left')


    startDelta=calc_time('2000-01-01T00:00:00')-race.start
    df_tim['timestamp']=df_tim.timestamp.apply(calc_time)
    df_tim['time']=df_tim.timestamp+startDelta
    df_tim['img']=df_tim.imagePath.isna() if 'imagePath' in df_tim else True
    display(df_tim.shape)
    self.df_tim=df_tim

  def uploadResult(self,df_tim,bibPattern=r"\d\d\d\d"):

    "Update header"
    self.updateStarttime()
    self.row_number=14
    df_upload=df_tim.loc[df_tim.bib.str.match(bibPattern),self.cols].fillna("")
    print(f"Updating for df_upload: {df_upload.shape}")
    df_upload.timestamp=df_tim.timestamp.astype(str)     #.dt.strftime("%Y-%m-%d %H:%M:%S.%f")#tz_convert() #

    df_upload.apply( self._updRow,
               axis=1)

  def _updRow(self,r):
    try:
      startcell=f"A{self.row_number}"
      self.row_number=self.row_number+1
      # print(startcell,r.bib)

      try: r.bib=int(r.bib)
      except: pass

      if r.imagePath!='':
        imagePath='https://storage.googleapis.com/run-pix.appspot.com/'+r.imagePath

      if r.waypoint in [5,10,'5','10']:
        r.waypoint+='k'

      # print(r.tolist())
      self.ss.append_row( r.tolist(),
                          table_range=startcell,
                          value_input_option='USER_ENTERED')
    except Exception as e:
      print(f"error {e!r}")

  def updateStarttime(self):
    mychoice=gc.open_by_url(cfg['sheets']['mychoice']['url'])

    self.ss=mychoice.worksheet(self.resultTab)
    self.ss.update_acell("b4",self._localeTimestamp(pd.Timestamp(race.timestamp['start'])))
    #worksheet.update([dataframe.columns.values.tolist()] + dataframe.values.tolist())
    #append_rows(values, value_input_option='RAW', insert_data_option=None, table_range=None, include_values_in_response=False)
    return
  def __repr__(self):
    # print((localeTimestamp(pd.Timestamp(race.timestamp['start']))))
    pd.Timestamp(race.timestamp['start']).tz_convert(cfg['tz']).strftime("%a %b %e %Y %H:%M:%S %z")


In [ ]:
# for d in df_images_bib_d[:20]:
# for d in df_images_bib_d[:20]:
#   for b in d['texts']:
#     _key = f"{d['timestamp']}_{b}"
#     _readingRef = race.fstore.document(f"races/{race.id}/readings/{_key}")
#     print( d['timestamp'],b,_key,_readingRef.get().exists)


## Export to my Choice

In [ ]:
resultTab='Race_MyCHOICE 2023-10' # @param


In [ ]:
"test"
mc=mychoice_result(race,resultTab)

## Search Readings

In [ ]:
" number of readings over timestamp"
import math
def bin_time(ts):
  return f"{ts.hour}:{math.floor(ts.minute/10)*10:02d}"

# df_readings=pd.DataFrame(race.readings)
# df_readings.
@widgets.interact(bib_search="")
def check_readings(bib_search):
  print(bib_search)
  if (bib_search):
    df_=mc.df_tim.query("bib.str.contains(@bib_search)")
  else:
    df_=mc.df_tim
  return df_.pivot_table(columns= 'waypoint',
                   index=[mc.df_tim['type'].fillna('-').str[:3],
                          mc.df_tim['timestamp'].apply(bin_time)],
                   values='userId',aggfunc='count').style.format(na_rep="-")

# check_readings()

## Podium lists

In [ ]:
"""list of podium bibs

- finished after start - not impl
- earliest finish
- for fathest distance
- valid bib
- photo validation needed

"""
def showTimeRec(r):
  r.time=r.time.strftime("%H:%M:%S")
  return r

df_podium=mc.df_tim[['Name', 'bib','time', 'waypoint','Gender','timestamp','type']
       ].sort_values( ['waypoint','Gender','bib','timestamp'], ascending=True
       ).drop_duplicates(
          ['bib', 'waypoint','Gender'],keep="first"
       ).groupby(['waypoint','Gender']
                               ).head(5)
df_podium.apply(showTimeRec,axis=1)

## Photo check for bib

In [ ]:
" Number of photos for each bib"

from IPython.display import Image,HTML,Markdown
def getImage(prefix,imagePath):
  return f"https://storage.googleapis.com/run-pix.appspot.com/{prefix}/{imagePath}"
def showImage(mc,imagePath)  :
  thb=getImage(f"thumbs/{mc.race.id}",imagePath)
  img=getImage(f"processed/{mc.race.id}",imagePath)
  display(HTML(f'<a href="{img}" target="_blank"><img src="{thb}"/></a>'))

# for bib_ in df_podium.bib[:3]:
@widgets.interact(bib_search="")
def getImageForBib(bib_search):
    if len(bib_search)<3: print( "To few letters")
    else:
      print( "Search for ",bib_search )
      for im in mc.race.images:
        if str(bib_search) in  im['texts']:# and i<20:
          showImage(mc,im['imagePath'])



# New Section

# New Section

In [ ]:
# for bib_ in df_podium.bib[:3]:
@widgets.interact(bib_found="0")
def getImageForNumberofBibs(bib_found):
      imagelist=[im for im in mc.race.images
                    if int(bib_found) == len(im['texts'])]
      print( "Search for ",bib_found,len(imagelist) )
      for i,im in enumerate(imagelist):
        if i<30:
          showImage(mc,im['imagePath'])

In [ ]:
# mc.uploadResult(df_tim,"^3\d\d\d$")

In [ ]:
searchImage="processed/mychoice23oct/2023-10-08T13:10:30.591Z~VENUE~breakingcoconut$gmail.com~KVP_4415.jpg"
pd.DataFrame(race.images).query(f"imagePath.str.contains('{searchImage[-10:]}')")
# race.images

In [ ]:
"being moved in to class = to be deleted"
raise


In [ ]:
df_

In [ ]:
"remove 10jul?"
mychoice_ws=gc.open_by_url(cfg['sheets']['mychoice']['url'])
mychoice_ss=mychoice_ws.worksheet(oldRaceTab)
df_apr23=pd.DataFrame(mychoice_ss.get('a9:f'))#pd.DataFrame(mychoice_ss.get('a10:f'),columns=mychoice_ss.get('a9:f9'))
df_apr23.columns=df_apr23.iloc[0,:]
df_apr23=df_apr23.iloc[1:,:]
df_apr23


CPU times: total: 109 ms
Wall time: 3.92 s

<Race:mychoice23apr bibs:0 Images:453
        status: ??
        blobs:[452, 453, 452]>
        

processed/mychoice23apr/2023-04-09T16:19:41.737Z~VENUE~jparagj$gmail.com~1P6A7290.jpg
processed/mychoice23apr/2023-04-09T16:19:41.737Z~VENUE~jparagj$gmail.com~1P6A7290.jpg

In [ ]:
" search for bib in readings - adhoc"
trace_bib='3183'
urlPref="https://storage.googleapis.com/run-pix.appspot.com/"#processed/{race.id}/"+"{x.imagePath}"
df_readings_bib=pd.DataFrame([d.to_dict()
              for d in race.fstore.collection(f"races/{race.id}/readings"
                  ).where('bib','==',trace_bib).stream()])
df_readings_bib['url']=df_readings_bib['imagePath'].apply(lambda x: urlPref+x if isinstance(x,str) else '')
df_readings_bib

In [ ]:
" ADHOC: check one image "
df_tim[~df_tim.imagePath.isna() & df_tim.imagePath.str.contains('0')]

In [ ]:
#@title
df_tim[' timestamp time bib Name Gender userId waypoint imagePath'.split()
    ].query("~Name.isna() & (Gender=='Male') & (userId.str.contains('mane'))",engine='python'
    ).sort_values("time"
    ).head(1000
           ).style.format({"timestamp":"{:%H:%M:%S}",
                           "time":"{:%H:%M:%S}",
                           "Gender":"{:.1s}"},na_rep="-")

In [ ]:
skip_wpts='VENUE venue'.split()
df_tim.query('~waypoint.isin(@skip_wpts)').pivot_table(index='Bib Name '.split(),#userId
                   columns="waypoint img".split(),
                   values="time",aggfunc="min").sort_values([('10K',True),('5K',True)])

## Read my choice 10 k sheets

In [ ]:
"-".join(list(('asd','ty')))

In [ ]:
"""Interactive time:
  Dependency: df_tim"""

display(Markdown('## List top'))
@widgets.interact(cat=widgets.Dropdown(value="All",options=['All']+df_tim.waypoint.fillna('').unique().tolist()),
                  gender=widgets.Dropdown(options=['All']+df_tim.Gender.fillna('').unique().tolist()),
                  bibSearch='')
def showRank(cat,gender,bibSearch):
  qry =  f"(waypoint=='{cat}')" if cat!='All' else "(bib!='')"
  qry += f"and (waypoint=='{gender}')" if gender!='All' else ""
  # qry += f"and ('{bibSearch}' in bib)" if bibSearch else ""
  df_=df_tim['timestamp waypoint bib type Gender userId'.split()].query(qry)
  df_piv= df_.pivot_table(index=['bib'],
                          columns='waypoint',
                          values="timestamp",aggfunc=["min","count"]
                          ).merge(ana['bibs'],
                                  how="left",
                                  left_on='bib',right_on="Bib").fillna('-')
  print(df_piv.columns)
  df_piv.columns=["-".join(list(obj)) if isinstance(obj, tuple) else obj for obj in df_piv.columns]
  return df_piv

# END NORMAL

In [ ]:
raise

# Periodic actions
## Load the bibs

In [ ]:
"  READ '1.myChoice Master' tab "  "COPIED ABOVE TOO"
def read_mychoice_master():
  gs=gc.open_by_url(cfg['sheets']['mychoice']['url'])
  # sheets = gs.worksheets()
  def rowToCol(df, column_row=0):
    df.columns=df.loc[column_row,:]
    df=df.drop(index=column_row)
    return df
  return rowToCol(pd.DataFrame(gs.worksheet('1.myChoice Master').get('A5:G')))
df_startlist=read_mychoice_master()

"  Check fields are matching etc. "

def checkMyChoiceBibData(df_startlist):
  display(F"Columns in clipboard: {df_startlist.columns.values}")
  if 'BIB Number' in df_startlist:
      df_startlist=df_startlist.rename(columns={'BIB Number': 'Bib', 'fullname': 'Name',  })
      df_startlist['Status']='From sheet'
      df_startlist['Race']='My Choice'
  _required_columns='Bib	Name	Status	Race Gender RegiId'.split()
  display(F"Columns Required: {_required_columns}")
  _found_columns = [_x for _x in _required_columns if  _x in df_startlist.columns]
  display("columns found ",_found_columns)
  if len(_found_columns)<len(_required_columns):
    print("PLEASE WAIT "+('X'*10+' ')*3)
  else:
    print(f"proceed to load {df_startlist.shape}")
  return df_startlist[_required_columns].reset_index(drop=True)

df_startlist=checkMyChoiceBibData(df_startlist)

In [ ]:
df_startlist

In [ ]:
" Load the bibs    -- comments out below"
# comments out below
# _=df_startlist.apply(race.saveBib,axis=1)

## Load results (external non DKD)?

In [ ]:
cfg.update(yaml.safe_load("""

results:
  url: https://docs.google.com/spreadsheets/d/13k7s153vFdL7_nswmW7WLFeszsiEhrfRG0Qi-AxmKUY/edit#gid=637535143
  tab: FINAL TIMING
  range: a5:p35
"""))

In [ ]:
"  READ '1.myChoice Master' tab "

gs=gc.open_by_url(cfg['results']['url'])
# sheets = gs.worksheets()
def rowToCol(df, column_row=0):
  df.columns=df.loc[column_row,:]
  df=df.drop(index=column_row)
  return df
df_results=rowToCol(pd.DataFrame(gs.worksheet(cfg['results']['tab']).get(cfg['results']['range'])))
display(Markdown(f"# reading {cfg['results']['tab']}"))

In [ ]:
"  Check fields are matching etc. "
cols="Bib	Name	Race	Rank	Start Time	Finish Time	Race Time	Gender	Category".split('\t')
# df_results[cols]
res_required_columns="""Bib,Name,Race,Rank,Gender,Start Time,Finish Time,Race Time,Category""".split(',')

def checkMyChoiceBibData(df_startlist):
  display(F"Columns in df_results: {df_startlist.columns.values}")
  if 'BIB Number' in df_startlist:
      df_startlist=df_startlist.rename(columns={'BIB Number': 'Bib', 'fullname': 'Name',  })


  display(F"Columns Required: {res_required_columns}")
  _found_columns = [_x for _x in res_required_columns if  _x in df_startlist.columns]
  display("columns found ",_found_columns)
  if len(_found_columns)<len(res_required_columns):
    print("PLEASE WAIT "+('X'*10+' ')*3)
  else:
    print(f"proceed to load {df_startlist.shape}")
  return df_startlist[res_required_columns].reset_index(drop=True)

df_results_1=checkMyChoiceBibData(df_results)

In [ ]:
def saveSesult(bibData):
  _bibData=bibData if isinstance (bibData,dict) else bibData.to_dict()
  _bibData['Bib']=str(_bibData['Bib'])
  _bibData['Name']=_bibData['Name'].upper()
  _path=f"races/{raceId}/result/{_bibData['Bib']}"
  runpix.fstore.document(_path).set(_bibData)
  print(f"loaded {_bibData['Bib']} {_bibData['Name']}")

In [ ]:
#df_results_1.apply(saveSesult,axis=1)

In [ ]:
" STOP HERE "
raise


In [ ]:
def searchImage(search):
    return [_ for _ in race.images if search in _['imagePath']]
# def setfireStorage
searchImage("2023-04-01T12:22:51.850Z")

In [ ]:
"""
    firebase Storage
"""
blobs=[blob for blob in bucket.list_blobs()]
df_blob=pd.DataFrame([{"id":b.id,"name":b.name,"blob":b,"size":b.size} for b in blobs])


In [ ]:
"""
    firestore
"""
def _f(x):
    _={"id":x.id,'ref':x}
    try: _.update(x.get().to_dict())
    except: pass
    return _
def fs2Df(collPath):
    return pd.DataFrame([_f(_) for _ in runpix.fstore.collection(collPath).list_documents()])

In [ ]:
"""
Utilities
"""
# from itertools import repeat
#runpix.delete_all_docs(runpix.fstore.collection('races').document('test').collection('readings'))


## other

In [ ]:
"query examples"
{_.id:_.to_dict() for _ in races.where('Location','==','GT').stream()}

In [ ]:
def addNewRace(id):
    raceData = {'Date': "YYYY-MM-DD",
                'Waypoints': ['venue', 'start', 'end'],
                'Name': 'Default Race Year Month',
                'Location': 'DC',
                'bibPattern': r'\d\d\d\d'
               }
    update_time, city_ref = races.document(id).set(raceData)
    print(f'Added document with id {city_ref.id}')

addNewRace('mychoice23apr')

In [ ]:
df_races.style

In [ ]:
def getDir(x):
    return pd.Series(x.split('/')[:-1]+3*[''])[:3]

def printBlobSummary(df):
    df['size_mb']=df['size']/10**6
    return pd.concat([df,df.name.apply(getDir)],axis=1)#).str.split("[\-/]",n=2,regex=True,expand=True)

printBlobSummary(df_blob).pivot_table(index=[0,1],values="size_mb",aggfunc=['sum','count']).style

# Reprocess Images (check)

In [ ]:
race

In [ ]:
"reprocess images - by search string"

searchString = "2023-07-09T04:53:45.692Z"
def reCopyBlob(b,dryrun=True):
  newName=race.blobs['uploads'][b]['name']
  blob=race.blobs['uploads'][b]['blob']
  print(b)
  runpix.bucket.copy_blob(blob,bucket,newName)

for i,b in enumerate(race.blobs['uploads']):
    if searchString in b :#and b.content_type=='image/jpeg':
        # newName="thumbs/mychoice23feb/"+b.name.split('/')[-1]
        # if i<691: continue
        print(i,sep="")
        reCopyBlob(b,dryrun=True)


In [ ]:
" Get the mismatches from uploads>processed>thumbs"
#  [race.blobs['uploads'][x]['name'].split("/")[-1][:-4] for x in race.blobs['uploads'] ]
def getMismatch(src='uploads',tgt='processed'):

  print(f"src {len(race.blobs[src])}, tgt {len(race.blobs[tgt])}",)

  target_list=[get_file_id(_) for _ in race.blobs[tgt]]
  # print(target_list[:10])
  # source_list=[race.blobs[src][_]['name'].split("/")[-1][:-4]+".jpg" for _ in race.blobs[src]]
  # display(source_list[:10],target_list[:10])
  mismatch=[x for x in race.blobs[src] \
            if not (get_file_id(x) in target_list)]
  # display(mismatch[:10])
  return mismatch

def get_file_id(blob_key):
    return race.blobs[blob_key.split("/")[1]][blob_key]['name'].split("/")[-1][:-4]+".jpg"


uploaded_blobids_but_not_processed = getMismatch("uploads","processed")
len(uploaded_blobids_but_not_processed)

In [ ]:
pd.DataFrame(uploaded_blobids_but_not_processed)

In [ ]:
for b in tqdm(uploaded_blobids_but_not_processed[:]):
  reCopyBlob(b,dryrun=False)

In [ ]:
df_images['texts'].explode().value_counts().reset_index().style

# Storage

# check blobs

In [ ]:
# for i, b in enumerate(race.blobs['processed']):
#     print (i,b)
from io import BytesIO
from PIL import Image
from base64 import b64decode
import IPython.display as Disp
class Blob:
  def __init__(self,blob):
    self.raceId=race.id
    self.blob=blob
  def show(self):
    base64_data = self.blob.download_as_bytes()
    display(Disp.Image(base64_data)) #b64decode
    return b64decode(base64_data)
  def signedURL(self):
    return self.blob.generate_signed_url(pd.Timestamp.today())

for b in race.blobs['uploads']:
  if 'O4A4959' in b:
    print(b)
    _b=Blob(race.blobs['uploads'][b]['blob'])
    display(_b.show())


 #"iVBORw0KGgoAAAANSUhEUgAABL ...  the rest of data "

In [ ]:
_b.blob.content_type

In [ ]:
import requests
import IPython.display as Disp
url = 'https://upload.wikimedia.org/wikipedia/commons/5/56/Kosaciec_szczecinkowaty_Iris_setosa.jpg'
print(requests.get(url).content)

In [ ]:
"Reprocess blobs"
# def newname(x):
#     arr=x.split('/')
#     arr[]
# gs://run-pix.appspot.com/thumbs/2023-02-12T01:25:41.084Z^venue^avinashmane$gmail.com^20230212_065538.jpg/2023-02-12T01:25:41.084Z^venue^avinashmane$gmail.com^20230212_065538.jpg
for i,b in enumerate(blobs):
    if 'thumbs/2023' in b.name :#and b.content_type=='image/jpeg':
        newName="thumbs/mychoice23feb/"+b.name.split('/')[-1]
        print(i,b.name,newName)
        bucket.copy_blob(b,bucket,newName)

In [ ]:

".generate_signed_url() needs date"
new_id="run-pix.appspot.com/thumbs/mychoice23feb/2023-02-12T01:25:41.084Z^venue^avinashmane$gmail.com^20230212_065538.jpg/1678882880166980"
blobs[72].content_type

In [ ]:
bucket.copy_blob()

# Bulk Bib Upload (old)

Data copied from clipboard from Excel:


In [ ]:
# raceId = 'werun2023'
display(Markdown(f"## Creating Bib list for {raceId}"))
bibs=runpix.fstore.collection(f'races/{raceId}/bibs')

In [ ]:
"Check Race details"
def dictUtf(mydict):
    return {k: str(v).encode("utf-8") for k,v in mydict.items()}

races.document(raceId).get().to_dict()

##  Copy the bib data to race (do not use)
Data copied from clipboard from Excel:

Columns copied are (case sensitive)
* Bib
* Name
* Status
* Race

In [ ]:
df_startlist=pd.read_clipboard()
display("columns in clipboard:",df_startlist.columns)
if 'BIB Number' in df_startlist:
    df_startlist=df_startlist.rename(columns={'BIB Number': 'Bib', 'fullname': 'Name',  })
    df_startlist['Status']='From sheet'
    df_startlist['Race']='My Choice'
_required_columns='Bib	Name	Status	Race'.split('\t')
_found_columns = [_x for _x in _required_columns if  _x in df_startlist.columns]
display("columns found ",_found_columns)
if len(_found_columns)<len(_required_columns): print("PLEASE WAIT "+('X'*10+' ')*3)


In [ ]:
df_startlist=df_startlist[_required_columns]

In [ ]:
"code"
bibtoDict=lambda x: x.to_dict()
bibtoDict(df_startlist.loc[0,:])  # test

def saveBibtoRace(bibData,bibsCollection):
    _bibData=bibtoDict(bibData)
    _bibData['Bib']=str(_bibData['Bib'])
    bibsCollection.document(_bibData['Bib']
                           ).set(_bibData)
saveBibtoRace(df_startlist.loc[0,:],bibsCollection=bibs,)  # test
# df_startlist.apply(, axis=1)


In [ ]:
" upload all bibs"
# df_startlist.apply(bibtoDict, axis=1)  #test
df_startlist.apply(lambda x: saveBibtoRace(x,bibs), axis=1)

In [ ]:
bibs.count().get()
# len(list( bibs.list_documents()))

# Bulk Upload images list_documents

In [ ]:
import yaml
import os
cfg={}
cfg.update(yaml.safe_load(r"""
uploads:
    werun2023:
        #path: D:\We Run 2023\JPEG
        path: D:\We Run 2023\Vaibhav\JPEG
        fileRange: ["","XX"]
        waypoint: venue
        userid: vaibhav
    mychoice23feb:
        path: D:\umesh\D K D  2023 2\Untitled Export
        waypoint: general
        userid: bcoconut
    mychoice23mar:
        path: D:\DKD-Parag\DKD 2023\DCIM\New folder
        waypoint: general
        userid: bcoconut
"""))


raceId='mychoice23mar'
cfg['uploads'][raceId]

In [ ]:
df_dir=dir2df(cfg['uploads'][raceId]['path'])
# df_dir['type root'.split()].value_counts()
print(f"{raceId} uploading from '{cfg['uploads'][raceId]['path']}',df_dir:{df_dir.shape}\n",df_dir.root.value_counts())

In [ ]:
"""
Manually rename to retrigger the processing...
- You can change waypoint from venue to general
"""
"* Storage: /uploads/race/time~wpt~user~loc~file    # uploaded images"
i=0
folder=f'uploads/{raceId}/'
import concurrent.futures
def getNewfileName(file,
                   user=cfg['uploads'][raceId]['userid'],
                   place=cfg['uploads'][raceId]['waypoint']):
    return "~".join([getFileIsoDate(file),place,user,getFileName(file)])

def uploadFile(root,name):
    path=os.path.join(root, name)
    new=folder+getNewfileName(path)
    # blob=bucket.blob(folder+name) #used when renaming to diff filenames
    blob=bucket.blob(new)

    if blob.exists():
        print(stats['files'],'blob exists/rewriting',name,new)
        blob.rewrite(blob)
    else:
        threads.append({new:executor.submit(blob.upload_from_filename,path)})
        print(stats['files'],f'uploading {path} to {new}', )
        # blob.upload_from_filename(path)


stats={"files":0,'upl':0}
prefix=f'processed/{raceId}'

all_blobs_l=[blob.name.split("~")[-1] for blob in bucket.list_blobs(prefix=prefix)]
executor=concurrent.futures.ThreadPoolExecutor(max_workers=5)
threads=[]

for lot in cfg['uploads'].keys():
    _dir=cfg['uploads'][lot]['path'].lower()
    print("raceId",lot,_dir,f"{len(all_blobs_l)} in {prefix}  ====================")
    # for root, dirs, files in os.walk(cfg['uploads'][lot]['path'], topdown=False):
    for i,d in df_dir.query(f"(type=='file') and (root.str.lower() == @_dir)"
                             ).iterrows():
        stats['files']+=1
        if not d.path in all_blobs_l:
            uploadFile(d.root,d.path)
            stats['upl']+=1
        else:
            print(prefix,d.path ,"found")
            pass
        # if (stats['files']>20): raise  #stopper
    print(f"{lot}: {stats['files']} files")
print(stats)

In [ ]:
"""
    find object from gcs

"""
display(Markdown("## rename blob"))
# bucket.blob('uploads/werun2023/_L3A3192.jpg').rewrite
# blob.rewrite?
bname='2023-03-13T19:23:14.739819~general~vaibhav~_L3A2997.jpg'
def renameBlob(bname):
    bucket.rename_blob(bucket.blob(bname),folder+bname)
# renameBlob(bname)
"use 2: existence check"
bucket.blob('error in getDownloadURL thumbs/werun2023/2023-03-13T19:23:42.882584~general~vaibhav~_L3A3007.jpg').exists()

## check storage file (which folder?)

In [ ]:
# check existance
# _path=r"processed/mychoice23feb/2023-02-12T01:28:29.364Z^venue^avinashmane$gmail.com^20230212_065828.jpg"
_path="processed/werun2023/2023-03-13T19:25:41.041091~general~vaibhav~_L3A3047.jpg"
[_,_raceId,_name]=_path.split("/")

print(_,_raceId,_name,)
for r in ['default',_raceId]:
    for t in 'processed thumbs uploads'.split():
        newName="/".join([t,r,_name])
        _blob=bucket.blob(newName)
        print(newName,_blob.exists() )


## check all images in the race

In [ ]:
df_images=fs2Df(f'races/{raceId}/images')
df_images.shape

In [ ]:
# df_images['imagePath'].value_counts()
def checkBlobs(x,types=['processed','thumbs']):
    return [bucket.blob(f'{typ}/{raceId}/{x}').exists()
                                        for typ in types]
def moveBlob(x,typ='thumbs'): #'processed',
    correctName=f'{typ}/{raceId}/{x}'
    if not bucket.blob(correctName).exists() :
        defaultThumb=bucket.blob(f'{typ}/default/{x}')

        if defaultThumb.exists():
            bucket.rename_blob(defaultThumb,f'{typ}/{raceId}/{x}')
            print(f"{correctName} not exists, moving from default")
        else:
            procBlob=bucket.blob(f'uploads/{raceId}/{x}')
            if procBlob.exists():
                print(f"{procBlob.name} rename the blob to retrigger functions")
                procBlob.rewrite(procBlob)

            else:
                procBlob.upload_from_filename(path4file+x.split("~")[-1])
                print(f"need to upload {x}")
    else:
        print(f"{correctName} exists")

path4file=r"D:\We Run 2023\JPEG\\"
display(raceId, path4file)
df_images['imagePath'].apply(lambda x: moveBlob(x))

In [ ]:
def deleteFSimages4missingBlobs(x,types=['processed','thumbs']):
    chk = checkBlobs(x.imagePath)
    if not any(chk):
        print(chk,x,x.imagePath)
        x.ref.delete()
df_images.apply(deleteFSimages4missingBlobs ,axis=1)

In [ ]:
df_blob.query("name.str.contains('G017')")

# One time tasks

## Clean blanks race readings

In [ ]:
"DElete all readings with no BIB...id ending with _"
for _r in race.readings[:1000]:
  # if isinstance(_r,str) and _r.bib=="":
  if _r['id'][-1]=='_':
    runpix.fstore.document(f"races/{raceId}/readings/{_r['id']}").delete()


### fix timing

In [ ]:
exifs=['ExifVersion', 'DateCreated', 'WhiteBalance', 'FocalLength', 'DigitalCreationTime',
       'DateTimeOriginal', 'OffsetTimeOriginal', 'SubSecTimeOriginal', 'ModifyDate',
       'ApproximateFocusDistance', 'OriginalDocumentID', 'format', 'ExposureMode',
       'Artist', 'ExposureCompensation', 'FlashCompensation', 'InstanceID', 'Lens',
       'Make', 'ExposureTime', 'LensModel', 'PreservedFileName', 'Flash', 'ISO']


def getExifData(img):
  imageData =runpix.fstore.document(f'races/{raceId}/images/{img}').get().to_dict()
  metadata=imageData['metadata']#
  # {_:imageData['metadata'][_] for _ in imageData['metadata'] if _ in exifs}
  # display(metadata)
  return metadata

def getCreateDate(metadata,default_ts):
  if 'DateTimeOriginal' in metadata:
    return metadata['DateTimeOriginal']#+metadata['OffsetTimeOriginal']
  if 'DateCreated' in metadata:
    return metadata['DateCreated']
  else:
    # print(metadata)
    return default_ts
img=ana['images'].loc[999,'imagePath']

pd.Timestamp(getCreateDate(getExifData(img),ana['images'].loc[999,'timestamp']))


In [ ]:
race.readings[300:304]

In [ ]:
" update reading with DateCreated "
if False:
  for i,_r in enumerate(race.readings[1900:2900]):
    try:
      if 'jparagj' in _r['userId'] and 'imagePath' in _r:
        img=_r['imagePath'].split('/')[-1]
        meta=getExifData(img)
        key=f"races/{raceId}/readings/{_r['id']}"

        if race.fstore.document(key).get().exists and meta['DateCreated']:
          race.fstore.document(key).update({u'timestamp':meta['DateCreated']})
          display(f"{i},{key},{meta['DateCreated']}")
    except:
      print('error with {_r["id"]}')


In [ ]:
"  end "

In [ ]:
mychoice23apr

In [ ]:
# allDocs=
""" COPY FIREBASE DATA """
for x in mychoice23apr.fstore.collection('races/mychoice23apr/images').stream():
    if '2023-04-02' in x.id:
        newDict = x.to_dict()
        newDict['metadata']['imagePath']=newDict['metadata']['imagePath'].replace('mychoice23apr','ahimsarun2023')
        # {k:(v.replace()
        #            if isinstance(v,str) else v)
        #            for (k,v) in .items()}
        mychoice23apr.fstore.document(f'races/ahimsarun2023/images/{x.id}').set(newDict)
        mychoice23apr.fstore.document(f'races/mychoice23apr/images/{x.id}').delete()
        print (x.id)
    else:
        print ('skipping', x.id)

In [ ]:
" move blobs from on race to another "
def copy_blob(name):
    # name="processed/mychoice23apr/2023-03-18T02:39:47.304Z~venue~avinashmane$gmail.com~capture.jpg"
    new_name=name.replace("mychoice23apr","ahimsarun2023")
    print("renaming",name,new_name)
    blob=mychoice23apr.storage.blob(name)
    if blob.exists():
        runpix.bucket.copy_blob(blob,runpix.bucket,new_name=new_name)
        pass

for r in "mychoice23apr".split():
    for folder in "processed".split():
        prefix=f"{folder}/{r}/2023-04-02"
        for i,blob in enumerate(mychoice23apr.storage.list_blobs(prefix=prefix)):
            # print(blob.name)
            copy_blob(blob.name)
            blob.delete()
            if i>100: break

In [ ]:
blob.delete()
Image(blob.download_as_bytes())

In [ ]:
copy_blob('processed/mychoice23apr/2023-04-02T00:43:14.291Z~venue~avinashmane$gmail.com~capture.jpg')

## Timing old vs new (unused)

In [ ]:
Markdown("# test")

In [ ]:
" bib in new not in old"
import numpy as np
skip_wpts = ["VENUE"] #@param {type:"raw"}
bibs_in_new=df_tim.query('~waypoint.isin(@skip_wpts) and ~Bib.isna()').bib.values
bibs_in_old=[str(_) for _ in df_apr23.bib.value_counts().index.levels[0].values if _ != 'RACE']
print(f"""
bibs_in_new {len(np.unique(bibs_in_new))} Unique / {len(bibs_in_new)}
bibs_in_old {len(bibs_in_old)}
common bibs {len([_ for _ in bibs_in_old if _ in bibs_in_new])}
only in new bibs {([_ for _ in bibs_in_new if not _ in bibs_in_old])}
""")

In [ ]:
bibs_in_new

# Other *May 2023 dkd*

In [ ]:
# df_tim.query("timestamp=='2023-05-14T01:30:03.831Z'")
import re
df_tim.type.str[:5].value_counts(),df_tim.shape
"fields to update at /races/{raceId}/readings/timestamp_bib/ "
attrs_="bib timestamp imagePath type userId waypoint"
def getReadingDict(row): ## bib_img.row
  try:
    (ts_,wpt_,userId_,type_,ext)= re.findall("(.*)~(.*)~(.*)~(.*)\.(png|jpg)",row.imagePath,)[0]
    d = { "bib":row.Bib,
          "timestamp":row.timestamp,
          "type":type_,
          "imagePath":f"processed/{raceId}/{row.imagePath.replace('.png','.jpg')}",
          "userId":userId_,
          "waypoint":wpt_,
        }
    return d
  except:
    print(row.index)
def getReadingFromBibImg(row):
  " uses raceId  "
  try:
    d = getReadingDict(row)
    key=d['timestamp']+'_'+d['bib'] # other option is row.timestamp which is later
    path=f"races/{raceId}/readings/{key}"

    # insert if entry does not exists
    if True :#runpix.fstore.document(path).get().exists == False:
      print( 'setting' ,path,'=',d)
      return runpix.fstore.document(path).set(d)
    else:
      print( 'already set' ,path,'=',d)
      return runpix.fstore.document(path).update(d)
  except Exception as e:
    print (f"error: {path} {e!r}")

In [ ]:
" fixing result for May 2023"
display(ana.keys())
ana['bib_img'][:].apply(getReadingFromBibImg, axis=1)#.values
# df_tim

In [ ]:
"Upload readings to excel"
mychoice.sheet1
# "Check images"
# ana['images']

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame(np.random.randint(0, 100, (100000, 6)))

# Register `pandas.progress_apply` and `pandas.Series.map_apply` with `tqdm`
# (can use `tqdm.gui.tqdm`, `tqdm.notebook.tqdm`, optional kwargs, etc.)
tqdm.pandas(desc="my bar!")

# Now you can use `progress_apply` instead of `apply`
# and `progress_map` instead of `map`
df.progress_apply(lambda x: x**2)
# can also groupby:
# df.groupby(0).progress_apply(lambda x: x**2)

In [ ]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()

class GSheet:
  def __init__(self,creds):
    self.gc = gspread.authorize(creds)
  def get_spreadsheets_like(self,match):
    return [f for f in self.gc.list_spreadsheet_files() if match in f['name']]


sheets=GSheet(creds).get_spreadsheets_like("My Choice")
display(sheets)
sheet = gc.open(sheets[0]['name'])

# get_all_values gives a list of rows.
rows = worksheet.get_all_values()
print(rows)

# Convert to a DataFrame and render.
import pandas as pd
pd.DataFrame.from_records(rows)

In [ ]:
# sheet.worksheets()
ws=sheet.worksheet('xRace_MyCHOICE 2023-04')

In [ ]:
rows=ws.get_all_values()
# get_all_values gives a list of rows.
# rows = worksheet.get_all_values()
print(rows)
# Convert to a DataFrame and render.
import pandas as pd
pd.DataFrame.from_records(rows)

# Image Screen/Search

In [ ]:
from PIL import Image, ImageDraw, ImageFont
from matplotlib.pyplot import imshow, figure
# from matplotlib import pyplot as plt
import io


# im.show()
class rpImage:
    # img0=f"races/{race.id}/images/{img0}"

    def __init__(self,img="races/mychoice23apr/images/2023-04-09T16:04:59.925Z~VENUE~jparagj$gmail.com~1P6A7093.jpg"):
        self.fsPath=img
        folder='uploads'
        self.imgBlobPath="/".join([folder,race.id,img.split("/")[-1]])
        display(self.imgBlobPath)
        self.fs=runpix.getDoc(img)

        self.blob=runpix.bucket.blob(self.imgBlobPath)
        self.im = Image.open(io.BytesIO(self.blob.download_as_bytes()))

    def mapxy(self,xy): return xy['x'],xy['y']
    def mapBoundingPoly(self,x):
        arr=[]
        for i in [0,2]:
            arr.append(self,mapxy(x['vertices'][i]))
        return arr
    def testMap(self):
        for _a in self.fs['textAnnotations']:
            print(_a['description'][:10],mapBoundingPoly(_a['boundingPoly']))

    def crop(self,pos,size,expand=1):
        topL1=self.im.size[0]*(pos[0]-expand)/100
        topL2=self.im.size[1]*(pos[1]-expand)/100
        botR1=self.im.size[0]*(pos[0]+size[0]+expand)/100
        botR2=self.im.size[1]*(pos[1]+size[1]+expand)/100
        img.im = img.im.crop((topL1,topL2,botR1,botR2))
        # print(topL1,topL2,botR1,botR2)
        return self

    def show(self,figsize = (12,8),inline=True):
        if inline:
            figure(figsize = (12,8))
            imshow(self.im)
        else:
            self.im.show()
    def drawAnnotations(self):
        # use a truetype font
        self.font = ImageFont.load_default()#ImageFont.truetype("arial.ttf", 40)
        self.draw = ImageDraw.Draw(self.im)
        # xy=[(x0, y0), (x1, y1)] or [x0, y0, x1, y1]
        # draw.rectangle([100,200,400,500], fill=None, outline=None, width=5)
        for i,_a in enumerate(self.fs['textAnnotations']):
            # print(_a['description'][:10],mapBoundingPoly(_a['boundingPoly']))
            # fnt = ImageFont.truetype("Pillow/Tests/fonts/FreeMono.ttf", 40)
            self.draw.rectangle(mapBoundingPoly(_a['boundingPoly']), fill=None, outline="#ffff0014", width=5)
            for i in range(4):
                # print(f"draw.line([{mapxy(_a['boundingPoly']['vertices'][i]),mapxy(_a['boundingPoly']['vertices'][(i+1)%4])}], fill=None, width=0, joint=None)")
                self.draw.line([mapxy(_a['boundingPoly']['vertices'][i]),mapxy(_a['boundingPoly']['vertices'][(i+1)%4])],
                               fill="#ffff0014", width=20,joint=None) #(255, 255, 0, 128)
            # draw text, half opacity
            try:
                self.draw.text(mapxy(_a['boundingPoly']['vertices'][0]), _a['description'],
                               font=self.font, fill="#ff000014")
            except:
                print('err')
                pass

In [ ]:
img=rpImage('races/mychoice23jul/images/2023-07-09T04:10:46.306Z~VENUE~jparagj$gmail.com~1P6A6486.jpg')

img.crop(pos=(30,50),size=(20,20)).show(figsize=(3,4))

In [ ]:
img.show()

In [ ]:
race.id

In [ ]:
arr=[]
for i,d in enumerate(runpix.fstore.collection(f"races/{race.id}/faces").stream()):
  x=d.to_dict()
  arr.append(x)
  print (i,py_.omit(x,"fd".split()))
  if i>2: break

In [ ]:
# arr[0]['fd'],arr[0]['image']
searchObj=arr[3]
img=rpImage(f'races/mychoice23jul/images/{searchObj["image"]}')

img.crop(pos=searchObj['pos'],size=searchObj['size'],expand=1).show(figsize=(2,2))

In [ ]:
!pip install firebase-admin --upgrade

In [ ]:
from google.cloud.firestore_v1.base_vector_query import DistanceMeasure
from google.cloud.firestore_v1.vector import Vector
from sklearn.metrics.pairwise import euclidean_distances
import pydash as py_

collection = race.fstore.collection(f"races/{race.id}/faces")

vector_query = collection.find_nearest(
    vector_field="fd",
    query_vector=searchObj['fd'],
    distance_measure=DistanceMeasure.EUCLIDEAN,
    limit=50,
    distance_result_field="vector_distance",
    # distance_threshold=4.5,
)

docs = vector_query.stream()

result=[]
for doc in docs:
    dat=doc.to_dict()
    x=py_.omit(dat,"fd".split())
    result.append(dat)
    print(f">> {doc.id}\n",x )

euclidean_distances( [x['fd'] for x in result] ,[arr[1]['fd']])
# result

In [ ]:
# [(int(x/5),x % 5) for x in range(20)]

In [ ]:
import matplotlib.pyplot as plt
w=5
h=10
f, axarr = plt.subplots(h,w,figsize=(10,10))
for i,dat in enumerate(result):
  img=rpImage(f'races/mychoice23jul/images/{dat["image"]}')

  img.crop(pos=dat['pos'],size=dat['size'],expand=1) #.show(figsize=(1,1))
    # display(img.im.show())
  axarr[int(i/w),i % w].imshow(img.im,)

In [ ]:
for dat in result:
    img=rpImage(f'races/mychoice23jul/images/{dat["image"]}')

    img.crop(pos=dat['pos'],size=dat['size'],expand=1).show(figsize=(1,1))
    display(img.im.show())

## Old Dashboard

In [ ]:
loadbibs_button = widgets.Button(description="Load Bibs from myChoice")
listraces_button=widgets.Button(description="List Races")
export_button = widgets.Button(description="Export (WIP)")
load_button = widgets.Button(description="Load Race")
create_button = widgets.Button(description="Create Race from")
clear_button = widgets.Button(description="Clear")
raceId_w = widgets.Text(value='mychoice23may', placeholder='enter raceId',    description='Race Id:',)
templateRaceId_w=widgets.Text(value='mychoice23apr',description='Template Race Id:',)
race=None
output = widgets.Output()

display(HBox([listraces_button,create_button,templateRaceId_w]),
        HBox([load_button,raceId_w,]),
        loadbibs_button,
        export_button,
        clear_button,
        output)

races=runpix.fstore.collection('races')
"list all races"
def list_races(_):
  df_races=runpix.fs2Df('races')
  _cols="Name   Date    id  Location status     photoStatus     ".split() #bibPattern Distances
  with output:
    display(df_races.sort_values('Date',ascending=False)[_cols])#.style.format(na_rep="-"))

def on_button_export(b):
    with output:
        print(f"Button clicked. {b!r}")
def on_button_loadrace(b):
    global race
    raceId = raceId_w.value
    with output:
      print(f"Loading race {raceId}")
      race= Race(raceId)
      print(race)
      race.getBlobs()
      # race.getBibs()
      # race.getimages()
      # race.getReadings()

      # x=[_.get().to_dict() for _ in self.fstore.collection(f'races/{self.id}/bibs').list_documents()]
      print(race)

def on_button_createrace(b):
    with output:
        Race(templateRaceId_w.value).copyRace(raceId_w.value)

def on_button_loadbibs(b):
  "  READ '1.myChoice Master' tab "
  def read_mychoice_master():
    gs=gc.open_by_url(cfg['sheets']['mychoice']['url'])
    # sheets = gs.worksheets()
    def rowToCol(df, column_row=0):
      df.columns=df.loc[column_row,:]
      df=df.drop(index=column_row)
      return df
    return rowToCol(pd.DataFrame(gs.worksheet('1.myChoice Master').get('A5:G')))

  "  Check fields are matching etc. "

  def checkMyChoiceBibData(df_startlist):
    display(F"Columns in clipboard: {df_startlist.columns.values}")
    if 'BIB Number' in df_startlist:
        df_startlist=df_startlist.rename(columns={'BIB Number': 'Bib', 'fullname': 'Name',  })
        df_startlist['Status']='From sheet'
        df_startlist['Race']='My Choice'
    _required_columns='Bib  Name    Status  Race Gender RegiId'.split()
    display(F"Columns Required: {_required_columns}")
    _found_columns = [_x for _x in _required_columns if  _x in df_startlist.columns]
    display("columns found ",_found_columns)
    if len(_found_columns)<len(_required_columns):
      print("PLEASE WAIT "+('X'*10+' ')*3)
    else:
      print(f"proceed to load {df_startlist.shape}")
    return df_startlist[_required_columns].reset_index(drop=True)

  with output:
    print("Loading bibs")
    df_startlist=read_mychoice_master()
    df_startlist=checkMyChoiceBibData(df_startlist)
    _=df_startlist.apply(race.saveBib,axis=1)

listraces_button.on_click(list_races  )
export_button.on_click(on_button_export)
create_button.on_click(on_button_createrace)
load_button.on_click(on_button_loadrace)
loadbibs_button.on_click(on_button_loadbibs)
clear_button.on_click(lambda x: output.clear_output())

## one time fix for virtual activities

In [ ]:
acts=runpix.getDocs(f"races/{race.id}/activities")

In [ ]:
# race.id,f"races/${race.id}"
for a_ in acts:
  parts=a_['id'].split('_')
  path_=f"races/{race.id}/activities/{a_['id']}"
  newVal={"activity":int(parts[2])}
  print(parts,path_,newVal)
  runpix.updateDoc(path_,newVal)